In [1]:
import pandas as pd
import numpy as np

print("Environment ready")

Environment ready


In [2]:
import sys
import pandas as pd
import numpy as np
import requests

print("Python:", sys.version)
print("Pandas:", pd.__version__)
print("NumPy:", np.__version__)
print("Requests:", requests.__version__)

Python: 3.12.2 (tags/v3.12.2:6abddd9, Feb  6 2024, 21:26:36) [MSC v.1937 64 bit (AMD64)]
Pandas: 2.2.2
NumPy: 1.26.4
Requests: 2.32.4


In [5]:
import os
import json
import requests
import pandas as pd
import world_bank_data as wb


# ============================================================
# CONFIGURACIÓN
# ============================================================

RAW_DIR = "../data/raw"
os.makedirs(RAW_DIR, exist_ok=True)

ISO3_COUNTRIES = ["MEX", "GBR", "EST", "CHL", "COL"]
ISO2_COUNTRIES = ["MX", "GB", "EE", "CL", "CO"]

# Nombre (como lo devuelve el World Bank) -> ISO3
ISO_MAP = {
    "Chile": "CHL",
    "Colombia": "COL",
    "Estonia": "EST",
    "United Kingdom": "GBR",
    "Mexico": "MEX",
}

# Rango de años en el que se busca el último dato disponible
START_YEAR = 2015
END_YEAR = 2025

TIMEOUT = 30
HEADERS = {"User-Agent": "Digitalization-Index-Data-Project/1.0"}

print("=" * 60)
print("INICIANDO EXTRACCIÓN PROGRAMÁTICA")
print("=" * 60)


# ============================================================
# FUNCIONES AUXILIARES
# ============================================================

def save_json(data, filepath):
    """Guarda un objeto Python como JSON."""
    with open(filepath, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)


def get_json(url, timeout=TIMEOUT):
    """Realiza GET y valida que la respuesta sea exitosa y JSON."""
    response = requests.get(url, timeout=timeout, headers=HEADERS)
    response.raise_for_status()
    return response.json()


def get_wb_indicator(code, value_name,
                     countries=ISO3_COUNTRIES,
                     start=START_YEAR, end=END_YEAR):
    """
    Descarga un indicador del World Bank y devuelve, por país,
    el último dato NO nulo dentro del rango de años indicado.

    Columnas de salida: country, iso3, year, <value_name>
    """
    s = wb.get_series(
        code,
        country=countries,
        date=f"{start}:{end}",
        simplify_index=True      # índice = (Country, Year)
    )

    # Se nombra la serie y se asignan columnas por NOMBRE, no por posición
    df = s.rename(value_name).reset_index()
    df.columns = ["country", "year", value_name]

    # Quitar años sin dato
    df = df.dropna(subset=[value_name])

    if df.empty:
        raise ValueError(
            f"Sin datos para {code} entre {start} y {end}"
        )

    df["year"] = df["year"].astype(int)

    # Último año disponible por país
    df = (
        df.sort_values(["country", "year"])
          .groupby("country", as_index=False)
          .tail(1)
          .reset_index(drop=True)
    )

    df["iso3"] = df["country"].map(ISO_MAP)

    return df[["country", "iso3", "year", value_name]]


# ============================================================
# FUENTE 1
# WORLD BANK
# (incluye TX.VAL.ICTG.ZS.UN, cuya fuente original es UNCTADstat,
#  y patentes de residentes, cuya fuente original es WIPO)
# ============================================================

print("\n[1/2] CONSULTANDO WORLD BANK API...")

wb_indicators = {
    "IT.NET.BBND.P2": "fixed_broadband_per100",
    "IT.NET.USER.ZS": "internet_users_pct",
    "TX.VAL.ICTG.ZS.UN": "ict_goods_exports_pct",     # fuente: UNCTADstat
    "GB.XPD.RSDV.GD.ZS": "rd_expenditure_gdp_pct",
    "IP.PAT.RESD": "patent_applications_resident",    # fuente: WIPO
}

wb_frames = {}

for code, name in wb_indicators.items():

    try:
        print(f"  -> Descargando {code}...")

        df = get_wb_indicator(code, name)

        output = os.path.join(RAW_DIR, f"wb_{name}.csv")
        df.to_csv(output, index=False, encoding="utf-8")

        wb_frames[name] = df

        print(f"  ✓ {name} descargado ({len(df)} registros)")
        print(df.to_string(index=False))

    except Exception as e:
        print(f"  ✗ ERROR en {code}: {e}")

# ------------------------------------------------------------
# Tabla consolidada (una fila por país, con el año de cada dato)
# ------------------------------------------------------------

if wb_frames:

    merged = pd.DataFrame({"iso3": ISO3_COUNTRIES})

    for name, df in wb_frames.items():
        part = df[["iso3", "year", name]].rename(
            columns={"year": f"{name}_year"}
        )
        merged = merged.merge(part, on="iso3", how="left")

    merged_path = os.path.join(RAW_DIR, "wb_all_indicators_latest.csv")
    merged.to_csv(merged_path, index=False, encoding="utf-8")

    print(f"\n  ✓ Tabla consolidada guardada: {merged_path}")

import os
import pandas as pd
import world_bank_data as wb


# ============================================================
# CONFIGURACIÓN
# ============================================================

RAW_DIR = "../data/raw"
os.makedirs(RAW_DIR, exist_ok=True)

ISO3_COUNTRIES = ["MEX", "GBR", "EST", "CHL", "COL"]

# Nombre (como lo devuelve el World Bank) -> ISO3
ISO_MAP = {
    "Chile": "CHL",
    "Colombia": "COL",
    "Estonia": "EST",
    "United Kingdom": "GBR",
    "Mexico": "MEX",
}

# Rango de años en el que se busca el último dato disponible
START_YEAR = 2015
END_YEAR = 2025

# código WB -> nombre de columna / archivo
WB_INDICATORS = {
    "IT.NET.BBND.P2": "fixed_broadband_per100",       # IND_01
    "IT.NET.USER.ZS": "internet_users_pct",           # IND_03
    "TX.VAL.ICTG.ZS.UN": "ict_goods_exports_pct",     # IND_05
    "IP.PAT.RESD": "patent_applications_resident",    # IND_07
    "GB.XPD.RSDV.GD.ZS": "rd_expenditure_gdp_pct",    # IND_08
}

print("=" * 60)
print("EXTRACCIÓN WORLD BANK (5 indicadores)")
print("=" * 60)


# ============================================================
# FUNCIÓN AUXILIAR
# ============================================================

def get_wb_indicator(code, value_name,
                     countries=ISO3_COUNTRIES,
                     start=START_YEAR, end=END_YEAR):
    """
    Descarga un indicador del World Bank y devuelve, por país,
    el último dato NO nulo dentro del rango de años indicado.

    Columnas de salida: country, iso3, year, <value_name>
    """
    s = wb.get_series(
        code,
        country=countries,
        date=f"{start}:{end}",
        simplify_index=True      # índice = (Country, Year)
    )

    df = s.rename(value_name).reset_index()
    df.columns = ["country", "year", value_name]

    df = df.dropna(subset=[value_name])

    if df.empty:
        raise ValueError(f"Sin datos para {code} entre {start} y {end}")

    df["year"] = df["year"].astype(int)

    df = (
        df.sort_values(["country", "year"])
          .groupby("country", as_index=False)
          .tail(1)
          .reset_index(drop=True)
    )

    df["iso3"] = df["country"].map(ISO_MAP)

    return df[["country", "iso3", "year", value_name]]


# ============================================================
# DESCARGA
# ============================================================

wb_frames = {}

for code, name in WB_INDICATORS.items():

    try:
        print(f"\n  -> Descargando {code}...")

        df = get_wb_indicator(code, name)

        output = os.path.join(RAW_DIR, f"wb_{name}.csv")
        df.to_csv(output, index=False, encoding="utf-8")

        wb_frames[name] = df

        print(f"  ✓ {name} ({len(df)} registros) -> {output}")
        print(df.to_string(index=False))

    except Exception as e:
        print(f"  ✗ ERROR en {code}: {e}")


# ============================================================
# TABLA CONSOLIDADA (una fila por país, con año de cada dato)
# ============================================================

if wb_frames:

    merged = pd.DataFrame({"iso3": ISO3_COUNTRIES})

    for name, df in wb_frames.items():
        part = df[["iso3", "year", name]].rename(
            columns={"year": f"{name}_year"}
        )
        merged = merged.merge(part, on="iso3", how="left")

    merged_path = os.path.join(RAW_DIR, "wb_all_indicators_latest.csv")
    merged.to_csv(merged_path, index=False, encoding="utf-8")

    print(f"\n  ✓ Tabla consolidada: {merged_path}")


print("\n" + "=" * 60)
print("EXTRACCIÓN FINALIZADA")
print("=" * 60)
print(f"\nArchivos guardados en: {os.path.abspath(RAW_DIR)}")
print(
    "\nPendientes por CSV manual: IND_02, IND_04 (ITU) e IND_06 (UNCTADstat). "
    "Ver 02_source_log.py para los nombres de archivo esperados."
)


# ============================================================
# FINAL
# ============================================================

print("\n" + "=" * 60)
print("EXTRACCIÓN FINALIZADA")
print("=" * 60)

print(f"\nArchivos guardados en: {os.path.abspath(RAW_DIR)}")

INICIANDO EXTRACCIÓN PROGRAMÁTICA

[1/2] CONSULTANDO WORLD BANK API...
  -> Descargando IT.NET.BBND.P2...
  ✓ fixed_broadband_per100 descargado (5 registros)
       country iso3  year  fixed_broadband_per100
         Chile  CHL  2024               23.731406
      Colombia  COL  2024               17.224416
       Estonia  EST  2024               35.352719
        Mexico  MEX  2024               21.736723
United Kingdom  GBR  2024               42.164828
  -> Descargando IT.NET.USER.ZS...
  ✓ internet_users_pct descargado (5 registros)
       country iso3  year  internet_users_pct
         Chile  CHL  2024           95.590243
      Colombia  COL  2024           79.345433
       Estonia  EST  2024           92.237962
        Mexico  MEX  2024           83.122338
United Kingdom  GBR  2024           95.472099
  -> Descargando TX.VAL.ICTG.ZS.UN...
  ✓ ict_goods_exports_pct descargado (5 registros)
       country iso3  year  ict_goods_exports_pct
         Chile  CHL  2024                   0

In [6]:
import os
import pandas as pd


RAW_DIR = "../data/raw"


# ============================================================
# SOURCE LOG
# ============================================================

source_log_data = [

    # ---------------- WORLD BANK (API) ----------------

    {
        "indicador_id": "IND_01",
        "nombre_oficial": "Fixed broadband subscriptions (per 100 people)",
        "categoria": "Infraestructura",
        "fuente": "World Bank (WDI)",
        "url_origen": "https://data.worldbank.org/indicator/IT.NET.BBND.P2",
        "unidad": "Por cada 100 hab",
        "metodo_obtencion": "API Python (world_bank_data)",
        "archivo_raw": "wb_fixed_broadband_per100.csv"
    },

    {
        "indicador_id": "IND_03",
        "nombre_oficial": "Individuals using the Internet (% of population)",
        "categoria": "Acceso / Uso",
        "fuente": "World Bank (WDI)",
        "url_origen": "https://data.worldbank.org/indicator/IT.NET.USER.ZS",
        "unidad": "% población",
        "metodo_obtencion": "API Python (world_bank_data)",
        "archivo_raw": "wb_internet_users_pct.csv"
    },

    {
        "indicador_id": "IND_05",
        "nombre_oficial": "ICT goods exports (% of total goods exports)",
        "categoria": "Económica Digital",
        "fuente": "World Bank (WDI)",
        "url_origen": "https://data.worldbank.org/indicator/TX.VAL.ICTG.ZS.UN",
        "unidad": "% exportaciones",
        "metodo_obtencion": "API Python (world_bank_data)",
        "archivo_raw": "wb_ict_goods_exports_pct.csv"
    },

    {
        "indicador_id": "IND_07",
        "nombre_oficial": "Patent applications, residents",
        "categoria": "Innovación",
        "fuente": "World Bank (WDI)",
        "url_origen": "https://data.worldbank.org/indicator/IP.PAT.RESD",
        "unidad": "Número de solicitudes",
        "metodo_obtencion": "API Python (world_bank_data)",
        "archivo_raw": "wb_patent_applications_resident.csv"
    },

    {
        "indicador_id": "IND_08",
        "nombre_oficial": "Research and development expenditure (% of GDP)",
        "categoria": "Innovación",
        "fuente": "World Bank (WDI)",
        "url_origen": "https://data.worldbank.org/indicator/GB.XPD.RSDV.GD.ZS",
        "unidad": "% del PIB",
        "metodo_obtencion": "API Python (world_bank_data)",
        "archivo_raw": "wb_rd_expenditure_gdp_pct.csv"
    },

    # ---------------- ITU DATAHUB (CSV MANUAL) ----------------

    {
        "indicador_id": "IND_02",
        "nombre_oficial": "4G mobile network coverage (% of population)",
        "categoria": "Infraestructura",
        "fuente": "ITU DataHub",
        "url_origen": "https://datahub.itu.int/data/",
        "unidad": "% población",
        "metodo_obtencion": "Descarga manual CSV (explorador de datos ITU)",
        "archivo_raw": "itu_4g_coverage.csv"
    },

    {
        "indicador_id": "IND_04",
        "nombre_oficial": "Mobile broadband basket (% of GNI per capita)",
        "categoria": "Asequibilidad",
        "fuente": "ITU DataHub",
        "url_origen": "https://datahub.itu.int/data/",
        "unidad": "% GNI per capita",
        "metodo_obtencion": "Descarga manual CSV (explorador de datos ITU)",
        "archivo_raw": "itu_mobile_broadband_basket.csv"
    },

    # ---------------- UNCTADstat (CSV MANUAL) ----------------

    {
        "indicador_id": "IND_06",
        "nombre_oficial": "Digitally deliverable services exports",
        "categoria": "Económica Digital",
        "fuente": "UNCTADstat",
        "url_origen": "https://unctadstat.unctad.org/",
        "unidad": "Según unidad publicada por UNCTADstat",
        "metodo_obtencion": "Descarga manual CSV (UNCTADstat)",
        "archivo_raw": "unctad_digitally_deliverable_services.csv"
    },
]


df_log = pd.DataFrame(source_log_data)


# ============================================================
# AÑO DE OBSERVACIÓN Y ESTADO DEL ARCHIVO
# ============================================================

YEAR_COLUMN_CANDIDATES = [
    "year", "Year", "YEAR", "time", "Time", "period", "Period",
    "ano", "año", "Año", "date", "Date", "TIME_PERIOD"
]


def get_year_range(filepath):
    """
    Devuelve el año (o rango min-max) observado en un CSV.
    Busca una columna de año por nombre; si no la encuentra devuelve None
    (habrá que completarlo a mano para los CSV descargados manualmente).
    """
    if not os.path.exists(filepath):
        return None

    try:
        df = pd.read_csv(filepath)
    except Exception:
        try:
            df = pd.read_csv(filepath, encoding="latin-1", sep=None,
                             engine="python")
        except Exception:
            return None

    col = next((c for c in YEAR_COLUMN_CANDIDATES if c in df.columns), None)
    if col is None:
        return None

    years = pd.to_numeric(df[col], errors="coerce").dropna()
    if years.empty:
        return None

    y_min, y_max = int(years.min()), int(years.max())
    return str(y_max) if y_min == y_max else f"{y_min}-{y_max}"


anos = []
estados = []

for _, row in df_log.iterrows():

    path = os.path.join(RAW_DIR, row["archivo_raw"])

    if os.path.exists(path):
        estados.append("Disponible")
        anos.append(get_year_range(path))
    else:
        estados.append("Pendiente")
        anos.append(None)

df_log["estado_archivo"] = estados
df_log["ano_observacion"] = anos


# ============================================================
# GUARDAR LOG
# ============================================================

output_path = "../source_log.csv"

df_log.to_csv(output_path, index=False, encoding="utf-8-sig")

print(f"✓ Archivo source_log.csv generado: {os.path.abspath(output_path)}")

print("\nResumen del source log:")
print(
    df_log[
        [
            "indicador_id",
            "nombre_oficial",
            "fuente",
            "estado_archivo",
            "ano_observacion",
        ]
    ].to_string(index=False)
)

pendientes = df_log[df_log["estado_archivo"] == "Pendiente"]

if not pendientes.empty:
    print("\nArchivos pendientes (colócalos en data/raw/):")
    for _, r in pendientes.iterrows():
        print(f"  - {r['archivo_raw']}  ({r['fuente']})")

✓ Archivo source_log.csv generado: C:\users\ramir\github_repos\digital-economy\source_log.csv

Resumen del source log:
indicador_id                                   nombre_oficial           fuente estado_archivo ano_observacion
      IND_01   Fixed broadband subscriptions (per 100 people) World Bank (WDI)     Disponible            2024
      IND_03 Individuals using the Internet (% of population) World Bank (WDI)     Disponible            2024
      IND_05     ICT goods exports (% of total goods exports) World Bank (WDI)     Disponible            2024
      IND_07                   Patent applications, residents World Bank (WDI)     Disponible            2021
      IND_08  Research and development expenditure (% of GDP) World Bank (WDI)     Disponible       2020-2024
      IND_02     4G mobile network coverage (% of population)      ITU DataHub      Pendiente            None
      IND_04    Mobile broadband basket (% of GNI per capita)      ITU DataHub      Pendiente            None
 